# 03. XGBoost + SHAP 분석

## 이 노트북이 하는 일
1. XGBoost로 **등급 예측 모델** 학습 (5-Fold CV)
2. **SHAP**으로 어느 카테고리가 등급을 결정하는지 분석
3. **Robustness 검증**: 여러 subset에서 결과 일관성 확인
4. 표/그림을 `results/` 에 저장

## 주요 결과
- **EA(에너지)가 등급 결정의 Top feature** (어느 subset에서든)
- CV 정확도 ~0.82, Weighted F1 ~0.82

In [ ]:
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

## 설정

In [ ]:
# Option A 데이터셋 사용 (Rule 점수 + LLM 리뷰 메타)
PARQUET = ROOT / 'data' / 'processed' / 'project_features_option_a.parquet'

# 결과 저장 경로
TABLE_DIR = ROOT / 'results' / 'tables'
FIG_DIR = ROOT / 'results' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

GRADE_ORDER = ['Certified', 'Silver', 'Gold', 'Platinum']
FEATURE_COLS = ['ratio_LT', 'ratio_SS', 'ratio_WE', 'ratio_EA',
                'ratio_MR', 'ratio_EQ', 'ratio_IP']

## 1단계: 데이터 로드

In [ ]:
df = pd.read_parquet(PARQUET)
print(f'전체: {len(df)}건')
print(f"등급 분포: {df['certification_level'].value_counts().to_dict()}")
print(f"버전 분포: {df['original_version'].value_counts().sort_index().to_dict()}")
print(f"LLM 리뷰됨: {df['has_llm_review'].sum()}건")

## 2단계: 공통 함수 정의

In [ ]:
def prepare_xy(data):
    """X, y 준비. ratio_SS NaN은 0으로 (v2009/v2.2 특성)."""
    ver_map = {'v2.0': 0, 'v2.2': 1, 'v2009': 2, 'v4': 3, 'v4.1': 4}
    data = data.copy()
    data['version_ord'] = data['original_version'].map(ver_map)
    data['log_area'] = np.log1p(data['gross_area_sqm'].fillna(0))
    
    cols = FEATURE_COLS + ['log_area', 'version_ord']
    X = data[cols].fillna(0)
    
    le = LabelEncoder()
    le.fit(GRADE_ORDER)
    y = le.transform(data['certification_level'])
    return X, y


def train_cv(X, y):
    """XGBoost 학습 + 5-Fold CV."""
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='mlogloss', verbosity=0,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(model, X, y, cv=cv, scoring='f1_weighted')
    model.fit(X, y)
    return model, cv_acc, cv_f1


def compute_shap(model, X):
    """SHAP TreeExplainer 계산."""
    expl = shap.TreeExplainer(model)
    sv = expl.shap_values(X)
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        sv = [sv[:, :, c] for c in range(sv.shape[2])]
    return sv


def shap_top_features(sv, X):
    """각 feature의 평균 |SHAP| 반환."""
    mean_abs = np.abs(np.stack(sv, axis=0)).mean(axis=(0, 1))
    return pd.Series(mean_abs, index=X.columns).sort_values(ascending=False)

## 3단계: 전체 460건 기본 분석

In [ ]:
mask_full = df['certification_level'].notna() & (df['certification_level'] != '')
X, y = prepare_xy(df[mask_full])

model, cv_acc, cv_f1 = train_cv(X, y)

print(f'N = {len(X)}')
print(f'CV 정확도: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'CV F1:     {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'각 Fold: {[f"{s:.4f}" for s in cv_acc]}')

In [ ]:
# SHAP 계산
sv_full = compute_shap(model, X)
top_full = shap_top_features(sv_full, X)

print('SHAP Top Feature:')
print(top_full.round(4))

## 4단계: 주요 figure 생성

In [ ]:
# Figure: SHAP bar plot (Top features)
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
top_full_ordered = top_full[::-1]
ax.barh(top_full_ordered.index, top_full_ordered.values, color='#1976D2')
ax.set_xlabel('Mean |SHAP|')
ax.set_title('SHAP Feature Importance (전체 460건)')
plt.tight_layout()
out = FIG_DIR / 'Figure1_shap_importance.png'
plt.savefig(out)
plt.close()
print(f'저장: {out}')

In [ ]:
# Figure: SHAP summary plot (beeswarm)
fig = plt.figure(figsize=(10, 6), dpi=150)
# 모든 클래스 SHAP 합산 (절댓값 평균)
combined_sv = np.mean([np.abs(s) for s in sv_full], axis=0)
shap.summary_plot(combined_sv, X, show=False, plot_size=(10, 6))
plt.tight_layout()
out = FIG_DIR / 'Figure2_shap_summary.png'
plt.savefig(out)
plt.close()
print(f'저장: {out}')

In [ ]:
# Figure: 등급별 카테고리 평균 달성률
grade_mean = df[mask_full].groupby('certification_level')[FEATURE_COLS].mean()
grade_mean = grade_mean.reindex(GRADE_ORDER)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
grade_mean.plot(kind='bar', ax=ax)
ax.set_ylabel('평균 달성률 (ratio)')
ax.set_title('등급별 카테고리 달성률 비교')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
out = FIG_DIR / 'Figure3_grade_factors.png'
plt.savefig(out)
plt.close()
print(f'저장: {out}')

## 5단계: Robustness 검증

여러 subset에서 Top SHAP feature가 일관되게 나오는지 확인.  
일관되면 결과 robust (흔들리지 않음).

In [ ]:
subsets = {
    '전체': mask_full,
    'credit_hit>0.7': mask_full & (df['credit_rule_hit_rate'].fillna(0) > 0.7),
    '구버전(v2.x, v2009)': mask_full & df['original_version'].isin(['v2.0', 'v2.2', 'v2009']),
    '신버전(v4, v4.1)': mask_full & df['original_version'].isin(['v4', 'v4.1']),
    'LLM 리뷰됨': mask_full & (df['has_llm_review'] == True),
}

results = []
for label, mask in subsets.items():
    sub = df[mask]
    if len(sub) < 20:
        continue
    X_s, y_s = prepare_xy(sub)
    m, acc, f1 = train_cv(X_s, y_s)
    sv_s = compute_shap(m, X_s)
    top_s = shap_top_features(sv_s, X_s)
    results.append({
        'subset': label,
        'N': len(X_s),
        'CV_Acc': round(acc.mean(), 4),
        'CV_F1': round(f1.mean(), 4),
        'Top1': top_s.index[0],
        'Top2': top_s.index[1],
        'Top3': top_s.index[2],
    })

robust_df = pd.DataFrame(results)
print(robust_df.to_string(index=False))

In [ ]:
# Table 저장
out = TABLE_DIR / 'Table4_robustness.csv'
robust_df.to_csv(out, index=False, encoding='utf-8-sig')
print(f'저장: {out}')

## 6단계: 주요 표 저장

In [ ]:
# Table 1: 데이터셋 사양
t1 = pd.DataFrame({
    '항목': ['총 건물', '등급: Gold', '등급: Silver', '등급: Platinum', '등급: Certified',
             'v4', 'v2009', 'v4.1', 'v2.2', 'v2.0'],
    '건수': [
        len(df),
        (df['certification_level'] == 'Gold').sum(),
        (df['certification_level'] == 'Silver').sum(),
        (df['certification_level'] == 'Platinum').sum(),
        (df['certification_level'] == 'Certified').sum(),
        (df['original_version'] == 'v4').sum(),
        (df['original_version'] == 'v2009').sum(),
        (df['original_version'] == 'v4.1').sum(),
        (df['original_version'] == 'v2.2').sum(),
        (df['original_version'] == 'v2.0').sum(),
    ]
})
t1.to_csv(TABLE_DIR / 'Table1_dataset.csv', index=False, encoding='utf-8-sig')

# Table 2: 모델 성능
t2 = pd.DataFrame([{
    'Model': 'XGBoost',
    'CV_Accuracy_mean': round(cv_acc.mean(), 4),
    'CV_Accuracy_std': round(cv_acc.std(), 4),
    'CV_F1_mean': round(cv_f1.mean(), 4),
    'CV_F1_std': round(cv_f1.std(), 4),
    'n_samples': len(X),
    'n_features': X.shape[1],
}])
t2.to_csv(TABLE_DIR / 'Table2_performance.csv', index=False, encoding='utf-8-sig')

# Table 3: SHAP Top 10
t3 = pd.DataFrame({
    'rank': range(1, len(top_full) + 1),
    'feature': top_full.index,
    'mean_abs_shap': top_full.values.round(4),
})
t3.to_csv(TABLE_DIR / 'Table3_shap_top.csv', index=False, encoding='utf-8-sig')

print('모든 표 저장 완료')
print(t3)

## 7단계: 결과 요약

In [ ]:
print('=' * 60)
print('  LEEDGRAPH 분석 결과 요약')
print('=' * 60)
print(f'데이터: {len(X)}건')
print(f'\n[모델 성능]')
print(f'CV 정확도:     {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'CV Weighted F1: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'\n[SHAP Top 3]')
for i, (feat, val) in enumerate(top_full.head(3).items(), 1):
    print(f'  {i}. {feat:15s}  {val:.4f}')
print(f'\n[Robustness] 모든 subset에서 Top1 = {robust_df["Top1"].iloc[0]}')
print(f"  {'✅ 일관' if robust_df['Top1'].nunique() == 1 else '⚠️ 불일치'}")

## 완료

- `results/tables/Table1~4.csv`: 주요 표
- `results/figures/Figure1~3.png`: 주요 그림

논문 작성 시 `outputs/reports/methodology_summary.md` 참고.